<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-LLM-Course/blob/main/06_The_Tokenizer_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

IN chapter 3, we looked at how to fine-tune a model on a given task. When we do that, we use the same tokenizer that the model was pretrained with -but what do we do when we want to train a model from scratch? In this cases, using a tokenizer that was pretrained on a corpus from another domain or language is typically suboptimal. For example, a tokenizer that's trained on an English corpus will perform poorly of Japanese texts because the use of spaces and punctuation is very different in the two languages.

In this chapter, you will learn how to train a brand new tokenizer on a corpus of texts, so it can then be used to pretrain a language model. This will all be done with the help of the `Tokenizers` https://github.com/huggingface/tokenizers library, which provides the "fast" tokenizers in the Transformers: https://github.com/huggingface/transformers library. We'll take a close look at the features that this library provides, and explore how the fast tokenizers differ from the "slow" versions.

Topics that will cover include:

  - How to train a new tokenizer similar to the one used by a given checkpoint on a new corpus of texts
  - The special features of fast tokenizers
  - The differences between the three main subword tokenization algorithms used in NLP today.
  - How to build a tokenizer from scratch with the tokenizer library and train it on some data

The techniques introduced in this chapter will prepare you for the section in Chapter 7 where we look at creating a language model for Python source code. Let's start by looking at what it means to "train" a tokenizer in the first place.

## Training a new tokenizer from a old one

If a language model is not available in the language you are interested in, or if your corpus is very different from the one your language model was trained on, you will most likely want to retrain the model from scratch using a tokenizer adapted to your data. That will require training a new tokenizer on your dataset. But what exactly does that mean? When we first looked at tokenizers in Chapter 2, we saw that most Transformers models use a `subword tokenization algorithm`. To identify which subwords are of interest and occurs most frequently in the corpus at hand, the tokenizer needs to take a hard look at all the texts in the corpus -a process we call `training`. The exact rules that govern this training depend of the type of tokenizer used, and we'll go over the three main algorithms later in this chapter.

### Assembling a corpus

There's a very simple API in Transformers that you can use to train a new tokenizer with the same characteristics as an existing one: `AutoTokenizer.train_new_from_iterator()`. To see this in action, let's say we want to train GPT-2 from scratch, but in a language other than English. Our first task will be to gather lots of data in that language in a training corpus. To provide examples everyone will be able to understand, we won't use a language like Russian or Chinese here, but rather a specialized English language: Python code.

The Datasets:https://github.com/huggingface/datasets library can help us assemble a corpus of Python source code. We'll use the usual `load_dataset()` function to download and cache the *CodeSearchNet* dataset. This dataset was created for the CodeSearchNet Challenge: https://forge.coreweave.com/wandb/github/CodeSearchNet and contains millions of functions from open source libraries on GitHub in several programming languages. Here, we will load the Python part of this dataset:


In [1]:
from datasets import load_dataset

# This can take a few minutes to load, so grab a coffee or tea while you wait!
raw_datasets = load_dataset("code-search-net/code_search_net", "python")

README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

python/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  522MB            

python/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

python/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.7MB            

python/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

python/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.7MB            

python/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

We can have a look at the training split to see which columns we have access to:

In [2]:
raw_datasets["train"]

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 412178
})

We can see the datasets separates docstrings from code and suggests a tokenization of both. Here, we'll just use the `whole_func_string` column to train our tokenizer. We can look at an example of one these functions by indexing into the `train` split:

In [3]:
print(raw_datasets["train"][123456]["whole_func_string"])

def oauth_token_create(self, data, **kwargs):
        "https://developer.zendesk.com/rest_api/docs/core/oauth_tokens#create-token"
        api_path = "/api/v2/oauth/tokens.json"
        return self.call(api_path, method="POST", data=data, **kwargs)


The first thing we need to do id transform the dataset into an `iterator` of lists of texts - for instance, a list of list of texts. Using lists of texts will enable our tokenizer to go faster (training on batches of texts instead of processing individuals texts one by one), and it should be an iterator if we want to avoid having everything in memory at once. If your corpus is huge, you will want to take advantage of the fact that Datasets does not load everything into RAM but stores the elements of the dataset on disk.

Doing the following would create a list of lists of 1,000 texts each, but would load everything in memory:


In [4]:
# Don't uncomment the following line unless your dataset is small!

# training_corpus = [raw_datasets["train"][i: i+1000]["whole_func_string"] for i in range(0, len(raw_datasets["train"]), 1000)]

Using Python generator, we can avoid Python loading anything into memory until it's actually necessary. To create such as generator, you just need to replace the brackets with parentheses:

In [5]:
training_corus = (
    raw_datasets["train"][i : i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

This line of code doesn't fetch any elements of the dataset; it just creates an object you can use in a Pyhton `for` loop. The texts will only be loaded when you need them (that is, when you're at the step of the `for` loop that requires them), and only 1,000 texts at a time will be loaded. This way you won't exhaust all your memory even if you are processing a huge dataset.

The problem with a generator object is that it can be only be used once. So, Instead of this giving us the first 10 digits twice:

In [6]:
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


We get them once and then an empty list:

That's why we define a function that returns a generator instead:

In [8]:
def get_training_corpus():
  return (
      raw_datasets["train"][i : i + 1000]["whole_func_string"]
      for i in range(0, len(raw_datasets["train"]), 1000)
  )
training_corpus = get_training_corpus()

You can also define your generator inside a `for` loop by using the `yield` statement:

In [9]:
def get_training_corpus():
  dataset = raw_datasets["train"]
  for start_idx in range(0, len(dataset), 1000):
    samples = dataset[start_idx : start_idx + 1000]
    yield samples["whole_func_string"]


which will produce the exact same generator as before, but allows you to use complex logic than you can in a list comprehension.